# 03 — Genetic Algorithm (GA) Feature Selection

Binary GA searches for a compact feature mask. This notebook uses the shared experiment pipeline so all optimizers receive the same data splits, classifiers, seeds, and evaluation rules.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Genetic Algorithm

In [2]:
# ----------------------------
# Binary Genetic Algorithm
# ----------------------------

def initialize_population(pop_size, n_features):
    return np.random.randint(0, 2, (pop_size, n_features))


def tournament_selection(population, fitness, k=3):
    idx = np.random.choice(len(population), k, replace=False)

    tournament_fitness = fitness[idx]

    winner = idx[np.argmin(tournament_fitness)]

    return population[winner].copy()


def crossover(parent1, parent2, pc=0.9):
    if random.random() > pc:
        return parent1.copy(), parent2.copy()

    point = random.randint(1, len(parent1)-2)

    child1 = np.concatenate((parent1[:point], parent2[point:]))
    child2 = np.concatenate((parent2[:point], parent1[point:]))

    return child1, child2


def mutation(individual, pm=0.05):
    child = individual.copy()

    for i in range(len(child)):
        if random.random() < pm:
            child[i] = 1 - child[i]

    if np.sum(child) == 0:
        child[random.randint(0, len(child)-1)] = 1

    return child


def run_ga(obj_func,
           n_features,
           pop_size=30,
           generations=50,
           pc=0.9,
           pm=0.05):

    population = initialize_population(pop_size, n_features)

    history = []

    best_solution = None
    best_fitness = np.inf

    for g in range(generations):

        fitness = np.array([obj_func(ind) for ind in population])

        idx = np.argmin(fitness)

        if fitness[idx] < best_fitness:
            best_fitness = fitness[idx]
            best_solution = population[idx].copy()

        history.append(best_fitness)

        # Do not create a final population that will never be evaluated.
        if g == generations - 1:
            break

        new_population = []

        while len(new_population) < pop_size:

            p1 = tournament_selection(population, fitness)
            p2 = tournament_selection(population, fitness)

            c1, c2 = crossover(p1, p2, pc)

            c1 = mutation(c1, pm)
            c2 = mutation(c2, pm)

            new_population.extend([c1, c2])

        population = np.array(new_population[:pop_size])

    return best_solution, best_fitness, history

## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [3]:
from utils.experiments import run_feature_selector


def ga_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_ga(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        generations=iterations,
        pc=0.9,
        pm=0.05,
    )


ga_results = run_feature_selector("GA", ga_runner)
ga_results.tail()


Saved: ('breast', 'svm', 0)


Saved: ('breast', 'svm', 1)


Saved: ('breast', 'svm', 2)


Saved: ('breast', 'svm', 3)


Saved: ('breast', 'svm', 4)


...

Saved: ('heart', 'xgboost', 1)


Saved: ('heart', 'xgboost', 2)


Saved: ('heart', 'xgboost', 3)


Saved: ('heart', 'xgboost', 4)


,Dataset,Classifier,Algorithm,Seed,ValidationFitness,Accuracy,Precision,Recall,F1,ROC_AUC,Features,SelectionRuntime,TestRuntime,SelectedFeatureNames,MaskFile,ConvergenceFile
25,heart,xgboost,GA,0,0.114809,0.788043,0.846154,0.754902,0.797927,0.892037,18,20.048789,0.094928,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""sex_Femal...",results/final/artifacts/heart__xgboost__ga__se...,results/final/artifacts/heart__xgboost__ga__se...
26,heart,xgboost,GA,1,0.135130,0.788043,0.838710,0.764706,0.800000,0.881994,15,19.849234,0.093568,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""cp_asympt...",results/final/artifacts/heart__xgboost__ga__se...,results/final/artifacts/heart__xgboost__ga__se...
27,heart,xgboost,GA,2,0.118189,0.760870,0.815217,0.735294,0.773196,0.872429,13,20.649972,0.095902,"[""chol"", ""thalch"", ""oldpeak"", ""cp_asymptomatic...",results/final/artifacts/heart__xgboost__ga__se...,results/final/artifacts/heart__xgboost__ga__se...
28,heart,xgboost,GA,3,0.121170,0.777174,0.814433,0.774510,0.793970,0.882413,7,20.474929,0.093582,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""cp_asympt...",results/final/artifacts/heart__xgboost__ga__se...,results/final/artifacts/heart__xgboost__ga__se...
29,heart,xgboost,GA,4,0.118189,0.798913,0.842105,0.784314,0.812183,0.878407,13,20.280767,0.094773,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""cp_asympt...",results/final/artifacts/heart__xgboost__ga__se...,results/final/artifacts/heart__xgboost__ga__se...
